## 1. Imports and Setup
First, necessary libraries and random seed for reproducibility

In [2]:
import os
import glob
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import albumentations as A 
from albumentations.pytorch import ToTensorV2
import cv2

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'Seeded everything with seed: {seed}')

## 2. Configuration

In [3]:
class Config:
    DATA_ROOT = '/Users/nguyenthientai/Documents/baseline_icpr_2026-main/train'
    IMG_HEIGHT = 32
    IMG_WIDTH = 128
    CHARS = "0123456789ABCDEFGHIKLMNOPQRSTUVWXYZ-"
    BATCH_SIZE = 64
    
    LEARNING_RATE = 0.0001
    EPOCHS = 50
    SEED = 42
    
    RESUME_TRAINING = False
    WEIGHTS_PATH = 'best_model.pth'
    
    if torch.backends.mps.is_available():
        DEVICE = torch.device("mps")
        print("MPS enabled")
    elif torch.cuda.is_available():
        DEVICE = torch.device("cuda")
        print("CUDA enabled")
    else:
        DEVICE = torch.device("cpu")
        print("CPU enabled")
        
    CHAR2IDX = {char: idx + 1 for idx, char in enumerate(CHARS)}
    IDX2CHAR = {idx + 1: char for idx, char in enumerate(CHARS)}
    NUM_CLASSES = len(CHARS) + 1  # +1 for CTC blank token
    VAL_SPLIT_FILE = "val_tracks.json"
    TEST_SPLIT_FILE = "test_tracks.json"
    VAL_SIZE = 2000
    TEST_SIZE = 2000


MPS enabled


## 3. Data Augmentation

In [1]:
def get_train_transforms():
    return A.Compose([
        A.Resize(height=Config.IMG_HEIGHT, width=Config.IMG_WIDTH),
        A.Affine(scale=(0.95, 1.05), translate_percent=(0.05, 0.05), rotate = (-5, 5), p=0.5, fill=128),
        A.Perspective(scale=(0.02, 0.05), p=0.3),
        A.RandomBrightnessContrast(p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
        A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(4, 8), hole_width_range=(4, 8), p=0.3),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])
    
def get_degradation_transforms():
    return A.Compose([
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3,7), p=1.0),
            A.Defocus(radius=(1,3), alias_blur=(0.1, 0.3), p=1.0),
            ], p=0.8),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
        ], p=0.8),
        
        A.ImageCompression(quality_range=(10, 50), p=0.5),
        A.Downscale(scale_range=(0.25, 0.5), p=0.5)
    ])
    
def get_val_transforms():
    return A.Compose([
        A.Resize(height=Config.IMG_HEIGHT, width=Config.IMG_WIDTH),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])

## 4. Dataset Class

In [ ]:
class AdvancedMultiFrameDataset(Dataset):
    def __init__(self, root_dir, mode='train'):
        self.mode = mode
        self.samples = []
        
        if mode == 'train':
            self.transform = get_train_transforms()
            self.degrade = get_degradation_transforms()
        else:
            self.transform = get_val_transforms()
            self.degrade = None
            
        print(f"[{mode.upper()}] Scanning: {root_dir}")
        abs_root = os.path.abspath(root_dir)
        search_path = os.path.join(abs_root, "**", "track_*")
        all_tracks = sorted(glob.glob(search_path, recursive=True))
        
        if not all_tracks:
            print("")